# 01. Data audit

**Goal:** understand what one row means in each table, check data quality, and define the target.
Nothing is modelled here. Everything downstream depends on getting this step right.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

RAW = "../data/raw"
PROCESSED = "../data/processed"

TARGET_ACTIONS = [
    "sub_car_claim_click", "sub_car_claim_submit_click", "sub_open_dialog_click",
    "sub_custom_question_submit_click", "sub_call_number_click",
    "sub_callback_submit_click", "sub_submit_success", "sub_car_request_submit_click",
]

## 1. Load once, save as Parquet
The raw CSVs are 0.4 GB (sessions) and 4.3 GB (hits). Parquet is columnar and compressed,
so later notebooks read only the columns they need in seconds.

In [2]:
sessions = pd.read_csv(f"{RAW}/ga_sessions.csv", dtype=str)
sessions["visit_number"] = sessions["visit_number"].astype(int)
sessions["visit_dt"] = pd.to_datetime(sessions["visit_date"] + " " + sessions["visit_time"])

hits = pd.read_csv(
    f"{RAW}/ga_hits.csv",
    usecols=["session_id", "hit_date", "hit_number", "hit_type", "hit_page_path",
             "event_category", "event_action"],
    dtype={"hit_number": "int32"},
)

sessions.to_parquet(f"{PROCESSED}/sessions.parquet", index=False)
hits.to_parquet(f"{PROCESSED}/hits.parquet", index=False)
print("sessions:", sessions.shape, "| hits:", hits.shape)

sessions: (1860042, 19) | hits: (15726470, 7)


## 2. Grain, what is one row?
* `sessions`: one row = one visit (`session_id`). A visitor (`client_id`) can have many visits.
* `hits`: one row = one event inside a visit.

In [3]:
print("unique session_id :", sessions["session_id"].nunique(), "of", len(sessions))
print("unique client_id  :", sessions["client_id"].nunique())
print("period            :", sessions["visit_dt"].min(), "→", sessions["visit_dt"].max())
print("hits per session  :", round(len(hits) / hits["session_id"].nunique(), 1))
print("\nsessions per client:")
print(sessions.groupby("client_id").size().describe(percentiles=[.5, .9, .99]).round(1))

unique session_id : 1860042 of 1860042


unique client_id  : 1391719
period            : 2021-05-19 10:00:00 → 2021-12-31 23:59:35


hits per session  : 9.1

sessions per client:


count    1391719.0
mean           1.3
std            1.9
min            1.0
50%            1.0
90%            2.0
99%            6.0
max          462.0
dtype: float64


## 3. Do the tables match?

In [4]:
s_ids = set(sessions["session_id"])
h_ids = set(hits["session_id"])
print("sessions with no hits :", len(s_ids - h_ids))
print("hit sessions not in sessions table:", len(h_ids - s_ids))

sessions with no hits : 127776
hit sessions not in sessions table: 2344


## 4. Missing and placeholder values
Google Analytics writes `(not set)` instead of an empty value, so NaN alone underestimates gaps.

In [5]:
quality = pd.DataFrame({
    "nan_%": sessions.isna().mean() * 100,
    "not_set_%": (sessions == "(not set)").mean() * 100,
    "n_unique": sessions.nunique(),
}).round(2).sort_values("nan_%", ascending=False)
quality

,nan_%,not_set_%,n_unique
device_model,99.12,0.00,104
utm_keyword,58.17,0.00,1219
device_os,57.53,0.02,13
device_brand,19.74,0.94,205
utm_adcontent,18.04,0.00,286
utm_campaign,11.81,0.00,412
utm_source,0.01,0.00,293
geo_city,0.00,4.20,2548
geo_country,0.00,0.06,166
device_browser,0.00,0.00,57


## 5. Target
A session is a **conversion** if it has at least one of the 8 target events
("leave a request", "order a call" …). Several target events in one visit still count as one.

In [6]:
is_target = hits["event_action"].isin(TARGET_ACTIONS)
print("target events by type:")
print(hits.loc[is_target, "event_action"].value_counts())

converted = set(hits.loc[is_target, "session_id"])
sessions["target"] = sessions["session_id"].isin(converted).astype(int)
print(f"\nconversion rate: {sessions['target'].mean():.2%}  "
      f"({sessions['target'].sum():,} of {len(sessions):,} sessions)")

target events by type:
event_action
sub_car_claim_click                 37928
sub_open_dialog_click               25870
sub_submit_success                  18439
sub_car_claim_submit_click          12359
sub_call_number_click                3653
sub_callback_submit_click            3074
sub_car_request_submit_click         2966
sub_custom_question_submit_click      619
Name: count, dtype: int64



conversion rate: 2.70%  (50,314 of 1,860,042 sessions)


In [7]:
sessions[["session_id", "target"]].to_parquet(f"{PROCESSED}/target.parquet", index=False)

## 6. Conversion over time
Checks for logging breaks or seasonality before we choose how to split train and test.

In [8]:
weekly = (sessions.set_index("visit_dt")
          .resample("W")["target"].agg(["size", "mean"])
          .rename(columns={"size": "sessions", "mean": "cr"}))
weekly["cr"] = (weekly["cr"] * 100).round(2)
weekly

,sessions,cr
visit_dt,,
2021-05-23,4219,2.54
2021-05-30,97980,6.14
2021-06-06,22872,6.10
2021-06-13,29964,5.11
2021-06-20,37845,4.20
2021-06-27,42354,3.59
2021-07-04,61003,2.65
2021-07-11,40716,2.84
2021-07-18,61824,1.97
